# Result · Preregistered hypotheses H1–H3

**In the paper ([arXiv:2610.10899](https://arxiv.org/abs/2610.10899)):** H1 in the Sec. V paragraph "Patching hardens narrowly", H2 (the BVCC non-inferiority margin) in "Fidelity", H3 in "Downstream".

The decision rules fixed in `preregistration.md`, evaluated on the E1 (hackability), E2 (agreement) and E3
(downstream) outputs. Recomputes all of E1, E2 and E3.

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e1', 'e2', 'e3_eval', 'e3_eval_r2']]
print(len(KEYS), "runs:", ", ".join(KEYS))

58 runs: e2_R0_s0, e2_R0_s1, e2_R0_s2, e1_R0_s0, e1_R3_s0, e1_R0_s1, e1_R3_s1, e1_R0_s2, e1_R3_s2, e1_baseline_noise_s0, e1_baseline_pgd_s0, e1_baseline_random_s0, e1_baseline_noise_s1, e1_baseline_pgd_s1, e1_baseline_random_s1, e1_baseline_noise_s2, e1_baseline_pgd_s2, e1_baseline_random_s2, e1_R1_s0, e1_R2_s0, e1_R1_s1, e1_R2_s1, e1_R1_s2, e1_R2_s2, e2_R1_s0, e2_R2_s0, e2_R3_s0, e2_baseline_noise_s0, e2_baseline_pgd_s0, e2_baseline_random_s0, e2_R1_s1, e2_R2_s1, e2_R3_s1, e2_baseline_noise_s1, e2_baseline_pgd_s1, e2_baseline_random_s1, e2_R1_s2, e2_R2_s2, e2_R3_s2, e2_baseline_noise_s2, e2_baseline_pgd_s2, e2_baseline_random_s2, e2_abl_tau0p1_s0_s0, e2_abl_margin0p1_s0_s0, e2_abl_headonly_s0_s0, e2_abl_margin1p0_s0_s0, e2_abl_prefinput_s0_s0, e3_eval_s0, e3_eval_s1, e3_eval_s2, e2_abl_tau0p5_s0_s0, e1_R3ms_s0, e2_R3ms_s0, e1_R3ms_s1, e2_R3ms_s1, e1_R3ms_s2, e2_R3ms_s2, e3_eval_R2_s0


In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    58
Name: count, dtype: int64

In [6]:
print((OUT / "stats.md").read_text())

H1: ASR(R0)-ASR(R3) per seed = [0.143, 0.005, 0.247], mean 0.131; supported = True (rule: mean ≥ 0.10 and all seeds > 0)

H1-transfer [grpo_dac]: ASR drop per seed [-0.274, 0.072, -0.317], mean -0.173; transfers = False

H1-transfer [pgd_wave]: ASR drop per seed [-0.003, 0.0, 0.0], mean -0.001; transfers = False

H1-transfer [random_encodec]: ASR drop per seed [0.006, 0.018, 0.012], mean 0.012; transfers = False

H1-transfer [transfer_r0_round1]: ASR drop per seed [0.213, 0.233, 0.308], mean 0.251; transfers = True

H2: BVCC-test sys-SRCC R3-R0 per seed = [-0.0012, 0.0018, -0.006], mean -0.0018; non-inferior = True (rule: mean ≥ -0.02)

H2 secondary: mean sys-SRCC over the 13 eval sets R0 0.745 vs R3 0.659; R3 significantly worse (Holm p<0.05) on 1/13 eval sets: vmc23/track2

H3: degraded external measures (mean over seeds) R0-reward 2.33 vs R3-reward 2.33; reward rise +0.263 / +0.269; supported = False


In [7]:
print(repro.diff_text(OUT / "stats.md", COMMITTED_OUT / "stats.md"))

identical to the committed file


In [8]:
# Numbers the paper quotes from this output: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('h1_', 'bvcc_sys_srcc_R3', 'e3_R0_n_external', 'e3_R3_n_external'))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; identical: {(cmp.abs_diff == 0).sum()}")
cmp.head(20)

6 numbers; max |diff| = 0.0000; identical: 6


,paper,reproduced,abs_diff
bvcc_sys_srcc_R3_minus_R0,-0.002,-0.002,0.0
e3_R0_n_external_worse,2.330,2.330,0.0
e3_R0_n_external_worse_2dp,2.330,2.330,0.0
e3_R3_n_external_worse,2.330,2.330,0.0
e3_R3_n_external_worse_2dp,2.330,2.330,0.0
h1_asr_drop,0.131,0.131,0.0
